# Optimisation

Servir un modèle revient à faire des compromis entre la vitesse du prefill, celle du decoding, la taille du contexte, la qualité des réponses et le nombre d'utilisateurs servis en même temps. llama-server dispose de plusieurs leviers avec lesquels il faut jouer pour servir correctement un modèle. Tout dépend entièrement du modèle, du matériel et de l'usage.

| Objectif | Mesure à surveiller | Leviers principaux |
|---|---|---|
| Faire tenir le modèle et son contexte en VRAM | tailles des buffers affichées par llama-server au démarrage | `-ngl`, `-c`, `-ctk` / `-ctv`, `-cmoe` / `-ncmoe` |
| Réduire l'attente avant le premier token | `prompt_ms`, `prompt_per_second` | tout en VRAM, `-ub`, cache de prompt |
| Générer plus vite | `predicted_per_second` | tout en VRAM, quantisation, speculative decoding |
| Servir plusieurs utilisateurs | débit total du serveur | `-np`, taille du contexte |

**Méthode** : changez un seul paramètre à la fois et mesurez avec les `timings` renvoyés par le serveur, comme le fait la fonction `evaluate` plus bas. Modifier plusieurs réglages d'un coup rend impossible de savoir lequel a eu quel effet.

Les mesures de ce notebook ont été faites avec **Qwen3.8-27B en UD-Q4_K_XL** (17,5 Go de poids) sur un GPU de 24 Go, avec le texte de `prompts/03_grand_prompt.md` comme prompt (environ 29 000 tokens), et un contexte de 131 072 tokens. Vos chiffres seront différents sur un autre matériel, mais les tendances restent les mêmes.

In [1]:
import polars as pl

from openai import OpenAI
import requests as rq

from pathlib import Path

from pprint import pprint

In [2]:
BASE_URL = '100.85.184.14:9931'
client = OpenAI(
    base_url=f'http://{BASE_URL}/v1',
    api_key='no-sk'
)

big_prompt = Path('prompts/03_grand_prompt.md').read_text('utf-8')

## Libérer de l'espace

La VRAM se partage entre plusieurs postes. Pour notre démo :

| Poste | Dépend de | Qwen3.8-27B UD-Q4_K_XL |
|---|---|---|
| Poids du modèle | taille du modèle, quantisation, `-ngl` | 16,3 Gio |
| KV cache | `-c`, `-ctk` / `-ctv`, architecture du modèle | de 1 à 16 Gio selon les réglages (voir plus bas) |
| Buffers de calcul | `-ub`, taille du vocabulaire | de l'ordre de 1 Gio |
| Drafter éventuel | méthode de speculative decoding | de quelques centaines de Mo à 2 Go |

Le pilote et le contexte CUDA prennent aussi quelques centaines de Mo : sur une carte de 24 Go, comptez environ 23 Gio réellement utilisables. llama-server affiche la taille de chaque buffer dans ses logs au démarrage, c'est la source la plus fiable.

Pour libérer de l'espace, vous avez plusieurs options. Essayez-les de préférence dans cet ordre : ajuster `-c` au besoin réel, quantiser le KV cache, puis seulement sortir des poids ou le cache du GPU.

### -ngl

`-ngl` (`--n-gpu-layers`) choisit le nombre de couches stockées en VRAM. Les couches restantes restent en RAM et sont calculées par le CPU. Moins il y a de couches en VRAM, plus le prefill et le decoding sont lents, mais cela permet soit de servir un modèle trop grand, soit d'augmenter la taille du contexte.

- `99` ou `all` met toutes les couches sur le GPU. `auto`, la valeur par défaut des versions récentes, laisse llama-server choisir selon la mémoire disponible.
- Notre modèle compte 64 couches pour 16,3 Gio : chaque couche retirée libère environ 250 Mio.
- Le ralentissement est rapide, car le decoding est limité par la vitesse de lecture des poids. La RAM (≈ 80 Go/s en DDR5) est environ 12 fois plus lente que la VRAM (≈ 1 000 Go/s). Avec seulement 6 couches sur 64 en RAM, le temps de lecture des poids double déjà à peu près.

### -c

`-c` (`--ctx-size`) définit la taille du contexte en tokens, et donc la taille du KV cache. `-c 0` reprend la taille d'entraînement du modèle : 262 144 tokens pour Qwen3.8-27B, ce qui ne tient pas sur 24 Go avec un cache en f16. Dimensionnez `-c` selon le besoin réel : le prompt le plus long plus la réponse.

Avec plusieurs slots (`-np`), le contexte est partagé. Historiquement, chaque slot reçoit `c / np` tokens. Avec `--kv-unified`, actif par défaut dans les versions récentes, le buffer est commun : un slot peut dépasser `c / np`, mais la somme de tous les slots ne dépasse jamais `-c`.

La taille du KV cache se calcule ainsi (le facteur 2 correspond à K et V) :

```
KV cache (octets) = 2 × couches_attention × n_ctx × n_head_kv × head_dim × octets_par_valeur
```

Qwen3.8-27B est un modèle **hybride** : sur ses 64 couches, seules 16 sont des couches d'attention classiques, avec 4 têtes KV de dimension 256. Les 48 autres sont des couches d'attention linéaire (Gated DeltaNet), dont l'état a une taille fixe qui ne grandit pas avec le contexte. En f16 :

```
2 × 16 × 4 × 256 × 2 octets = 64 Kio par token
```

À titre de comparaison, un modèle dense classique comme Qwen3-32B (attention sur ses 64 couches, 8 têtes KV de dimension 128) consomme 256 Kio par token, soit quatre fois plus. Ces valeurs se lisent dans le `config.json` du modèle sur Hugging Face ou dans les métadonnées du GGUF.

| `-c` | f16 | q8_0 | q4_0 |
|---:|---:|---:|---:|
| 65 536 | 4,0 Gio | 2,1 Gio | 1,1 Gio |
| 131 072 | 8,0 Gio | 4,3 Gio | 2,3 Gio |
| 262 144 | 16,0 Gio | 8,5 Gio | 4,5 Gio |

### -ctk et -ctv

À l'instar des poids, il est possible de quantiser le KV cache : `-ctk` (`--cache-type-k`) pour les clés K, `-ctv` (`--cache-type-v`) pour les valeurs V. Cela réduit son empreinte mémoire mais peut avoir une influence sur la qualité des réponses, surtout sur les longs contextes.

| Type | Bits par valeur | Taille par rapport à f16 | Impact sur la qualité |
|---|---:|---:|---|
| `f16` (défaut) | 16 | 100 % | référence |
| `q8_0` | 8,5 | 53 % | quasi nul |
| `q5_1` / `q5_0` | 6 / 5,5 | 38 % / 34 % | faible |
| `q4_1` | 5 | 31 % | modéré |
| `q4_0` / `iq4_nl` | 4,5 | 28 % | visible sur les longs contextes et le raisonnement |

Nous préférerons quantiser V avant K. Les clés servent au produit avec les requêtes Q, dont le résultat passe par le softmax : l'exponentielle amplifie les petites erreurs, et les clés contiennent des valeurs extrêmes qui se quantisent mal. Les erreurs sur V, elles, sont atténuées par la moyenne pondérée de l'attention.

Par ordre de préférence :

1. `-ctk q8_0 -ctv q8_0` divise le cache par deux quasiment sans perte. C'est le premier réflexe.
2. `-ctk q8_0 -ctv q4_0` est un bon compromis.
3. `-ctk q4_0 -ctv q4_0` est un dernier recours : testez la qualité sur vos propres tâches.

Points d'attention :

- Quantiser V nécessite Flash Attention. Elle est activée par défaut quand le matériel le permet (`-fa auto`), `-fa on` la force.
- Sur CUDA, les combinaisons mixtes (K et V de types différents) nécessitent de compiler llama.cpp avec `-DGGML_CUDA_FA_ALL_QUANTS=ON`. Sans cette option, l'attention retombe sur le CPU et devient très lente.
- Un drafter a son propre KV cache, réglable avec `-ctkd` et `-ctvd`.

**Exemple avec Qwen3.8-27B** : poids (16,3 Gio) + KV cache + buffers (~1 Gio), sur une carte de 24 Go.

| KV cache | Total à `-c 131072` | Tient ? | Total à `-c 262144` | Tient ? |
|---|---:|---|---:|---|
| f16 | ~25,3 Gio | non | ~33,3 Gio | non |
| q8_0 | ~21,6 Gio | oui | ~25,8 Gio | non |
| K q8_0, V q4_0 | ~20,6 Gio | oui | ~23,8 Gio | non, de justesse |
| q4_0 | ~19,6 Gio | oui | ~21,8 Gio | oui |

Les commandes de ce notebook utilisent `-c 131072`. Sans quantisation, le cache ne tiendrait pas en VRAM. `q8_0` suffirait, et `q4_0` laisse plus de 3 Gio de marge, par exemple pour un drafter. Pour le contexte complet du modèle, seul `q4_0` passe.

### -kvo et -nkvo

Dans le vocabulaire de llama.cpp, *offload* signifie « envoyer sur le GPU ». `-kvo` (`--kv-offload`, le défaut) garde le KV cache en VRAM. `-nkvo` (`--no-kv-offload`) le laisse en RAM, et l'attention est alors calculée par le CPU.

Cela libère toute la place du KV cache, mais l'impact sur la vitesse est certain, surtout pour le decoding : à chaque token généré, toute l'attention sur le contexte se calcule côté CPU. C'est un dernier recours, après avoir réduit `-c` et quantisé le cache. Il reste utile pour des traitements par lots où la vitesse compte peu.

### -cmoe et -ncmoe

Ces options ne concernent que les modèles **MoE** (*Mixture of Experts*), comme Qwen3.5-35B-A3B : 35 milliards de paramètres au total, dont seulement 3 milliards actifs par token. Qwen3.8-27B est un modèle dense, elles ne s'y appliquent pas.

Dans un MoE, les experts représentent l'essentiel des poids, mais chaque token n'en utilise que quelques-uns. L'idée est de garder sur le GPU ce qui sert à chaque token (attention, KV cache, couches partagées) et de laisser les experts en RAM. Le CPU ne lit alors que les experts actifs, ce qui limite fortement le ralentissement.

| Option | Effet |
|---|---|
| `-cmoe` (`--cpu-moe`) | tous les experts restent en RAM |
| `-ncmoe N` (`--n-cpu-moe N`) | seuls les experts des N premières couches restent en RAM |

**Méthode** : démarrez avec `-ngl 99 -cmoe`, puis remplacez par `-ncmoe N` en diminuant N tant que tout tient en VRAM. Sur un MoE, c'est généralement bien plus rapide qu'un `-ngl` partiel, qui sort des couches entières, attention comprise.

### --context-shift

Cette option ne libère pas de VRAM : elle change ce qui se passe quand le contexte est plein.

- **Désactivée** (défaut) : la génération s'arrête avec `finish_reason == "length"`, et un prompt trop long est refusé.
- **Activée** : le serveur supprime les tokens les plus anciens et décale le KV cache pour continuer à générer.

La génération peut alors continuer indéfiniment, mais le modèle oublie le début de la conversation sans prévenir, consignes comprises. L'option n'est pas disponible sur tous les modèles, notamment ceux qui ont des couches récurrentes comme Qwen3.8. Le plus souvent, il vaut mieux la laisser désactivée et gérer l'historique côté client (résumé, fenêtre glissante), comme dans l'exercice 02.

**Configuration de référence** : tout en VRAM, KV cache quantisé en q4_0.

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -kvo \
--port 9931 --host 0.0.0.0 
```

In [ ]:
def evaluate(client, prompt: str, max_tokens: int = 512):
    r = client.chat.completions.create(
        model='local',
        messages=[
            {'role': 'user', 'content': f'Resume et explique moi ce texte: <document>\n{prompt}\n</document>'}
        ]
        , max_tokens=max_tokens
    )

    return pl.from_dict(r.model_extra['timings'])

evaluate(client, big_prompt)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second
i64,i64,f64,f64,f64,i64,f64,f64,f64
0,28976,21605.504,0.745634,1341.139739,512,13867.714,27.138384,36.848178


Même configuration, mais le KV cache reste cette fois en RAM (`-nkvo`) :

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -nkvo \
--port 9931 --host 0.0.0.0 
```

In [4]:
evaluate(client, big_prompt)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second
i64,i64,f64,f64,f64,i64,f64,f64,f64
0,28976,31069.957,1.072265,932.605089,512,63779.379,124.812875,8.011994


### Mesure : KV cache en VRAM ou en RAM

| | KV cache en VRAM (`-kvo`) | KV cache en RAM (`-nkvo`) | Écart |
|---|---:|---:|---:|
| Prefill (tok/s) | 1 341 | 933 | −30 % |
| Decoding (tok/s) | 36,8 | 8,0 | ÷4,6 |

Le decoding est le plus touché : à chaque token généré, il faut relire tout le KV cache pour l'attention, et ce travail se fait désormais en RAM, sur le CPU. Avec notre prompt de 29 000 tokens, cela représente beaucoup de données à relire pour chaque token. Le prefill souffre moins, car le calcul des couches, qui domine cette phase, reste sur le GPU.

## Accélérer le prefill et le decoding

Si vous souhaitez accélérer le prefill et le decoding, vous avez plusieurs options. La première, nous l'avons vu, est de tout mettre en VRAM : plus il y a de choses en VRAM, plus ça ira vite.

Rappelez-vous que le prefill est limité par la puissance de calcul et le decoding par la bande passante mémoire (voir `support/prefill_and_decoding.html`). Les leviers ne sont donc pas les mêmes pour les deux phases.

| Levier | Effet sur le prefill | Effet sur le decoding | Commentaire |
|---|---|---|---|
| Tout en VRAM (`-ngl 99`, `-kvo`) | fort | très fort | premier réflexe |
| Quantisation plus forte des poids | moyen | fort | moins d'octets à lire par token, au prix de la qualité |
| Flash Attention (`-fa on`) | moyen | faible | surtout utile sur les longs contextes |
| Micro-batch plus grand (`-ub`) | fort | aucun | voir ci-dessous |
| Speculative decoding | légèrement négatif | fort | voir la section Drafter |
| Cache de prompt | très fort si le préfixe est commun | aucun | voir la fin du notebook |

### Batch et micro-batch

Pendant le prefill, le prompt est découpé en morceaux, et chaque morceau traverse le modèle en une seule passe.

- **`-b`** (`--batch-size`, défaut 2048) est la taille *logique* : le nombre maximal de tokens que le serveur soumet au modèle en un appel. Avec plusieurs slots, les tokens de plusieurs requêtes sont regroupés dans ce batch.
- **`-ub`** (`--ubatch-size`, défaut 512) est la taille *physique* : le batch logique est redécoupé en micro-batchs de cette taille, réellement calculés ensemble par le GPU. `-ub` doit être inférieur ou égal à `-b`.

Plus le micro-batch est grand, plus chaque poids lu en VRAM sert à de nombreux tokens, et plus le prefill est rapide, jusqu'à saturer la puissance de calcul. En contrepartie, le buffer de calcul grossit avec `-ub`.

Pendant le decoding avec un seul utilisateur, chaque passe ne contient qu'un token (ou quelques-uns avec le speculative decoding). Ces deux paramètres n'ont donc pas d'effet sur la vitesse de génération.

| Usage | `-b` | `-ub` |
|---|---|---|
| Usage général | 2048 (défaut) | 512 (défaut) |
| Longs prompts (documents, RAG), VRAM disponible | 2048 à 4096 | 1024 à 2048 |
| VRAM très juste | 512 | 128 à 256 |
| Embeddings et reranking | au moins la taille de l'entrée la plus longue | égal à `-b`, car chaque entrée doit tenir dans un seul micro-batch |

Privilégiez des puissances de 2 (128, 256, 512, 1024, 2048).

**Petit batch** : `-b 256 -ub 64`. Nous passons `max_tokens=0` à `evaluate`, car seul le prefill nous intéresse ici.

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -kvo \
-b 256 -ub 64 \
--port 9931 --host 0.0.0.0 
```

In [5]:
evaluate(client, big_prompt, 0)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second
i64,i64,f64,f64,f64,i64,f64,f64,f64
0,28976,30019.338,1.036007,965.24447,1,0.001,0.0,0.0


**Grand batch** : `-b 4084 -ub 512`.

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -kvo \
-b 4084 -ub 512 \
--port 9931 --host 0.0.0.0 
```

In [6]:
evaluate(client, big_prompt, 0)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second
i64,i64,f64,f64,f64,i64,f64,f64,f64
0,28976,21726.425,0.749808,1333.675467,1,0.001,0.0,0.0


#### Résultats

| Réglage | `prompt_ms` | Prefill (tok/s) |
|---|---:|---:|
| `-b 256 -ub 64` | 30 019 | 965 |
| `-b 2048 -ub 512` (défauts, première mesure) | 21 606 | 1 341 |
| `-b 4084 -ub 512` | 21 726 | 1 334 |

C'est `-ub` qui compte : passer de 64 à 512 accélère le prefill d'environ 39 %. Augmenter `-b` au-delà de 2048 avec le même `-ub` n'apporte rien pour une seule requête, l'écart de moins de 1 % relève du bruit de mesure. Pour aller plus loin, il faudrait augmenter `-ub` (1024, 2048), ce que llama-bench permet de tester rapidement.

### Drafter : le speculative decoding

En decoding, le GPU passe l'essentiel de son temps à attendre la mémoire : ses unités de calcul sont presque inoccupées. Le speculative decoding exploite ce calcul inutilisé.

1. Un *drafter*, rapide, propose k tokens.
2. Le modèle principal vérifie les k propositions en une seule passe, comme un mini-prefill : il lit ses poids une seule fois pour k + 1 positions.
3. Il garde les propositions jusqu'à la première qu'il refuse et place son propre token à cet endroit.

Une passe de vérification coûte à peine plus qu'un decoding normal, mais elle rapporte plusieurs tokens. Le texte produit suit la distribution du modèle principal : seule la vitesse change. La section 4 de `support/prefill_and_decoding.html` anime ce mécanisme.

#### Les méthodes disponibles (`--spec-type`)

| Méthode | Drafter | Fichier en plus | Principe |
|---|---|---|---|
| `draft-simple` | un petit modèle de la même famille, avec le même tokenizer | oui (`-md`) | propose les tokens un par un |
| `draft-eagle3` | une couche légère entraînée pour le modèle cible | oui | lit les états internes du modèle principal |
| `draft-mtp` | les têtes MTP (*Multi-Token Prediction*) intégrées au modèle | non | le modèle a été entraîné à prédire aussi les tokens suivants |
| `draft-dflash`, `draft-dspark` | un petit modèle de diffusion par blocs | oui (`-md`) | propose un bloc entier de tokens en une seule passe |
| `ngram-*` | aucun | non | recopie des suites déjà vues dans le contexte |

#### Ce que ça coûte en VRAM

Le drafter prend un peu de place, à compter dans le budget VRAM. Avec un KV cache déjà au plus juste, ajouter un drafter peut obliger à réduire `-c`.

| Méthode | VRAM supplémentaire pour Qwen3.8-27B |
|---|---|
| `draft-mtp` | la couche MTP, environ 1/64 du modèle, soit de l'ordre de 250 Mo en Q4, plus son KV cache |
| `draft-dflash` avec DFlash2 en Q8_0 | 2,06 Go de poids, plus son KV cache |
| `ngram-*` | quasi nulle |

#### Quand l'utiliser, quand l'éviter

| Favorable | Défavorable |
|---|---|
| Un seul utilisateur ou une faible charge : le calcul du GPU est inoccupé | Beaucoup d'utilisateurs en parallèle : le GPU est déjà occupé, et les propositions rejetées prennent la place de tokens utiles |
| Texte prévisible : code, JSON, extraction, résumé qui reprend le texte source | Texte créatif, température élevée |
| Réponses longues | Réponses très courtes, ou tâches dominées par le prefill (classification, réponse en un mot) |
| VRAM disponible pour le drafter | VRAM déjà au plus juste |

#### Code ou texte libre

Le taux d'acceptation dépend surtout de la prévisibilité du texte. En code, beaucoup de tokens sont presque imposés : syntaxe, indentation, noms de variables déjà définis, parenthèses fermantes. Le drafter les devine facilement, et le gain peut dépasser ×2. En texte libre, et plus encore en texte créatif, plusieurs suites sont plausibles, et le drafter se trompe plus souvent.

Pour ordre d'idée, la fiche du drafter DFlash2 annonce environ 5 tokens par passe sur des problèmes de mathématiques (GSM8K). Sur notre résumé d'un essai philosophique en français, nous obtenons moins de 3 tokens par passe.

#### Lire les métriques

Avec un drafter, les `timings` contiennent deux champs de plus : `draft_n` (tokens proposés) et `draft_n_accepted` (tokens acceptés).

```
taux d'acceptation = draft_n_accepted / draft_n
tokens par passe   = predicted_n / (predicted_n - draft_n_accepted)
```

Le chiffre qui compte reste `predicted_per_second` : un bon taux d'acceptation ne sert à rien si le drafter coûte trop cher.

Premier essai avec MTP et 2 tokens proposés par passe :

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -kvo \
-b 4084 -ub 512 \
--spec-type draft-mtp --spec-draft-n-max 2 \
--port 9931 --host 0.0.0.0 
```

In [7]:
evaluate(client, big_prompt)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second,draft_n,draft_n_accepted
i64,i64,f64,f64,f64,i64,f64,f64,f64,i64,i64
0,28976,22844.973,0.78841,1268.375323,512,9530.409,18.650507,53.617846,507,256


Avec beaucoup de tokens proposés par passe (`--spec-draft-n-max 10`) :

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -kvo \
-b 4084 -ub 512 \
--spec-type draft-mtp --spec-draft-n-max 10 \
--port 9931 --host 0.0.0.0 
```

In [8]:
evaluate(client, big_prompt)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second,draft_n,draft_n_accepted
i64,i64,f64,f64,f64,i64,f64,f64,f64,i64,i64
0,28976,22994.301,0.793564,1260.138327,512,13068.489,25.574342,39.10169,1881,320


Avec Dflash

Avec DFlash2 : un drafter séparé (`-md`) qui propose un bloc entier de tokens en une seule passe.

```sh
llama-server \
-m ./models/qwen3.8-27b/llama/Qwen3.8-27B-UD-Q4_K_XL.gguf \
-np 1 -ngl 99 -c 131072 \
-ctk q4_0 -ctv q4_0 -kvo \
-b 4084 -ub 512 \
--spec-type draft-dflash --spec-draft-n-max 10 \
-md ./models/qwen3.8-27b/llama/Qwen3.8-27B-DFlash2-Q8_0.gguf \
--port 9931 --host 0.0.0.0 
```

In [10]:
evaluate(client, big_prompt)

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second,draft_n,draft_n_accepted
i64,i64,f64,f64,f64,i64,f64,f64,f64,i64,i64
0,28976,24291.857,0.838344,1192.827704,512,11723.011,22.941313,43.589484,1349,317


### Bilan du speculative decoding

| Configuration | `draft_n` | Acceptés | Taux | Tokens par passe | Decoding (tok/s) | Gain |
|---|---:|---:|---:|---:|---:|---:|
| Sans drafter | – | – | – | 1 | 36,8 | – |
| MTP, n-max 2 | 507 | 256 | 50 % | 2,0 | 53,6 | ×1,46 |
| MTP, n-max 10 | 1 881 | 320 | 17 % | 2,7 | 39,1 | ×1,06 |
| DFlash2, n-max 10 | 1 349 | 317 | 23 % | 2,6 | 43,6 | ×1,18 |

- **MTP avec 10 propositions ne rapporte presque plus rien** : ×1,06 contre ×1,46 avec 2 propositions. Qwen3.8 n'a qu'une couche MTP, entraînée à prédire le token suivant. En l'enchaînant 10 fois, les erreurs s'accumulent : le taux d'acceptation s'effondre, et chaque proposition coûte une passe MTP et une position à vérifier.
- **DFlash2 accepte plus de tokens par passe que MTP avec 2 propositions** (2,6 contre 2,0) mais reste plus lent : son drafter pèse 2 Go, et chaque passe vérifie 8 positions.
- **DFlash2 plafonne** : 1 349 propositions pour 195 passes, soit environ 7 par passe malgré `--spec-draft-n-max 10`. llama-server ramène la valeur à la taille de bloc sur laquelle le drafter a été entraîné.
- **Le prefill ralentit légèrement**, de 1 341 à 1 190-1 270 tok/s selon la méthode, car le drafter doit lui aussi traiter le prompt.

#### Valeurs à retenir

| Méthode | `--spec-draft-n-max` | Pourquoi |
|---|---|---|
| `draft-mtp` | 2 à 3 (défaut 3) | au-delà, l'acceptation s'effondre |
| `draft-dflash` | 7 pour DFlash2 Qwen3.8, la valeur recommandée par la fiche du modèle | le bloc est proposé en une seule passe, et une valeur plus haute est de toute façon plafonnée |

Validez toujours sur votre propre tâche : les valeurs optimales dépendent du modèle, du texte et du matériel.

### Température et cache de prompt

Si maintenant nous jouons avec d'autres paramètres, comme la température, la vitesse du speculative decoding change aussi.

#### Pourquoi la température joue

Le drafter propose ses tokens les plus probables. Le modèle principal, lui, tire chaque token avec ses propres paramètres d'échantillonnage, puis compare : une proposition n'est acceptée que si elle correspond au token qu'il a tiré.

- À température 0, les deux choisissent le favori et tombent souvent d'accord.
- Plus la température monte, plus la distribution s'aplatit et plus le tirage s'écarte du favori. Les rejets augmentent et le gain diminue.

Les filtres `top_k` et `min_p` atténuent l'effet, car ils éliminent les candidats improbables avant le tirage.

#### Le cache de prompt

La cellule ci-dessous envoie trois fois le même prompt, ce qui montre aussi l'effet du cache. Après chaque requête, le slot garde le KV cache du prompt. Si la requête suivante commence par les mêmes tokens, llama-server les réutilise et ne calcule que la suite : c'est la différence entre `cache_n` (tokens repris du cache) et `prompt_n` (tokens calculés).

| Paramètre | Où | Défaut | Rôle |
|---|---|---|---|
| `cache_prompt` | requête (`extra_body`) | `true` | réutilise le préfixe commun avec la requête précédente du slot |
| `--cache-reuse N` | serveur | 0 (désactivé) | réutilise aussi des morceaux d'au moins N tokens situés plus loin dans le prompt, en décalant le KV cache. Utile quand seul un passage au milieu du prompt change. Valeur repère : 256. Nécessite un modèle dont le cache peut être décalé |
| `--cache-ram N` (`-cram`) | serveur | 8192 Mio | cache de prompts en RAM : conserve les états de conversations précédentes et les recharge quand le même préfixe revient. -1 : illimité, 0 : désactivé |
| `--ctx-checkpoints N` (`-ctxcp`) | serveur | 32 | pour les modèles hybrides ou à fenêtre glissante, comme Qwen3.8 : leur état ne peut pas être tronqué n'importe où, le serveur enregistre donc des points de reprise pour pouvoir réutiliser un préfixe |
| `--slot-save-path` | serveur | désactivé | sauvegarde et restaure le cache d'un slot sur disque, via l'API `/slots` |

Le cache ne fonctionne que sur un préfixe identique au token près. Placez la partie stable (system prompt, documents) au début et la partie variable (question) à la fin. Un élément variable en tête du prompt, comme la date ou l'heure, invalide tout le cache.

In [ ]:
temperature = (0, 0.5, 1.7)

data = []
for t in temperature:
    r = client.chat.completions.create(
        model='local',
        messages=[
            {'role': 'user', 'content': f'Resume et explique moi ce texte: <document>{big_prompt}</document>'}
        ]
        , max_tokens=2048
        , temperature=t
        , extra_body={'cached_prompt': True}
    )

    metrics = r.model_extra['timings']
    metrics.update({'temperature': t})
    data.append(metrics)

pl.from_dicts(data).sort('draft_n_accepted')

cache_n,prompt_n,prompt_ms,prompt_per_token_ms,prompt_per_second,predicted_n,predicted_ms,predicted_per_token_ms,predicted_per_second,draft_n,draft_n_accepted,temperature
i64,i64,f64,f64,f64,i64,f64,f64,f64,i64,i64,f64
28972,4,146.658,36.6645,27.274339,2048,50967.481,24.898623,40.162864,5918,1200,1.7
28972,4,147.643,36.91075,27.092378,2048,43089.416,21.050032,47.505865,5005,1331,0.5
28972,4,566.708,141.677,7.058309,2048,38077.181,18.601456,53.759232,4454,1409,0.0


#### Résultats

| Température | `cache_n` | `prompt_n` | `prompt_ms` | Taux d'acceptation | Decoding (tok/s) |
|---:|---:|---:|---:|---:|---:|
| 0 | 28 972 | 4 | 567 | 32 % | 53,8 |
| 0,5 | 28 972 | 4 | 148 | 27 % | 47,5 |
| 1,7 | 28 972 | 4 | 147 | 20 % | 40,2 |

- **Température** : plus elle monte, plus le taux d'acceptation baisse, de 32 % à 20 %. La génération ralentit d'autant : environ 12 % plus lente à 0,5 et 25 % plus lente à 1,7 qu'à température 0.
- **Cache** : les trois requêtes reprennent 28 972 tokens du cache laissé par la requête précédente et n'en calculent que 4. Le premier token arrive en 150 à 570 ms, contre 24 s pour le même prompt calculé en entier dans la mesure DFlash2 : entre 40 et 165 fois plus tôt.
- **Piège de lecture** : quand le cache sert, `prompt_per_second` n'a plus de sens. Il est calculé sur les 4 tokens recalculés seulement, d'où un débit apparent très faible.

## llama-bench

llama-bench est l'outil de benchmark fourni avec llama.cpp, installé avec les autres binaires. Il mesure séparément les deux phases sur des tokens aléatoires, et répète chaque test (5 fois par défaut) pour donner une moyenne et un écart-type :

- **`ppN`** (*prompt processing*) : prefill d'un prompt de N tokens, `pp512` par défaut.
- **`tgN`** (*text generation*) : decoding de N tokens, `tg128` par défaut.

Son intérêt : chaque paramètre accepte une liste de valeurs séparées par des virgules, et llama-bench teste toutes les combinaisons. C'est bien plus rapide que de relancer le serveur pour chaque réglage.

| Option | Défaut | Rôle |
|---|---|---|
| `-m` | | modèle, peut être répété pour comparer plusieurs modèles |
| `-p` | 512 | taille du prompt pour le test de prefill (0 : pas de test) |
| `-n` | 128 | nombre de tokens générés pour le test de decoding (0 : pas de test) |
| `-pg pp,tg` | | test combiné : prefill puis génération |
| `-d` | 0 | profondeur : nombre de tokens déjà présents dans le KV cache avant le test |
| `-ngl`, `-fa`, `-b`, `-ub`, `-ctk`, `-ctv`, `-nkvo`, `-ncmoe` | | mêmes rôles que dans llama-server |
| `-r` | 5 | nombre de répétitions |
| `-o` | `md` | format de sortie : `md`, `csv`, `json`, `jsonl`, `sql` |

Quelques exemples :

```sh
# Mesure de base
llama-bench -m modele.gguf -ngl 99 -fa on

# Trouver le meilleur micro-batch pour le prefill
llama-bench -m modele.gguf -ngl 99 -fa on -p 4096 -n 0 -b 4096 -ub 128,256,512,1024,2048

# Vitesse selon le remplissage du contexte, KV cache en f16 puis en q4_0
llama-bench -m modele.gguf -ngl 99 -fa on -d 0,16384,32768
llama-bench -m modele.gguf -ngl 99 -fa on -ctk q4_0 -ctv q4_0 -d 0,16384,32768

# Combien de couches pouvons-nous sortir du GPU ?
llama-bench -m modele.gguf -fa on -ngl 99,60,56,48 -p 0 -n 128

# KV cache en VRAM ou en RAM
llama-bench -m modele.gguf -ngl 99 -fa on -nkvo 0,1

# Export en CSV, à analyser ensuite avec polars
llama-bench -m modele.gguf -ngl 99 -fa on -ub 256,512,1024 -o csv > bench.csv
```

La sortie est un tableau avec une ligne par combinaison : le modèle, sa taille, le backend, les paramètres qui varient, le test (par exemple `pp512`, `tg128` ou `pp512 @ d16384`) et la vitesse en tokens par seconde, avec son écart-type.

> **Attention** : llama-bench teste toutes les combinaisons. `-ctk q8_0,q4_0 -ctv q8_0,q4_0` donne 4 configurations, dont 2 mixtes qui nécessitent un build avec `GGML_CUDA_FA_ALL_QUANTS`. Le nombre de tests grimpe aussi très vite : 3 paramètres à 4 valeurs, c'est 64 configurations répétées 5 fois.

### Les limites de llama-bench

llama-bench mesure le moteur de calcul, pas le serveur. Une bonne partie de ce que nous avons vu dans ce notebook lui échappe.

| Ce que llama-bench ne mesure pas | Pourquoi | Alternative |
|---|---|---|
| Speculative decoding (MTP, DFlash, drafter) | aucune option de drafter, et sur des tokens aléatoires un drafter ne devinerait rien | requêtes réelles au serveur, comme `evaluate` |
| Plusieurs utilisateurs (`-np`) | un seul flux à la fois | `llama-batched-bench`, ou des requêtes parallèles au serveur |
| Cache de prompt, `--cache-reuse`, context shift | ce sont des fonctions du serveur | requêtes réelles |
| Paramètres d'échantillonnage (température, etc.) | aucun vrai texte n'est généré | requêtes réelles |
| Qualité des réponses | seule la vitesse est mesurée | évaluation sur vos propres tâches, `llama-perplexity` pour un indicateur global |
| Consommation de VRAM | elle n'est pas affichée, une configuration trop grosse échoue simplement | logs de llama-server, `nvidia-smi` |
| Multimodal (images) | pas de projecteur `mmproj` | requêtes réelles |

Pour le débit avec plusieurs utilisateurs, `llama-batched-bench` simule N séquences en parallèle :

```sh
llama-batched-bench -m modele.gguf -ngl 99 -fa on -c 16384 -npp 512 -ntg 128 -npl 1,2,4,8
```

En pratique, llama-bench sert à dégrossir les réglages liés au matériel (`-ub`, `-ngl`, types de KV cache, `-fa`). Les réglages qui dépendent du texte et de l'usage (drafter, cache, température) se valident ensuite avec de vraies requêtes au serveur, comme dans ce notebook.